### **Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend **


## Task 4.1 - One validated call

In [107]:
# Step 1 — Install the libraries

!pip install -q google-genai pydantic



In [81]:
# Step 2 — Set up the Gemini call

import json
from getpass import getpass
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.1-flash-lite"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You classify call-centre notes about missed clinic appointments.
Return only the requested JSON object.
"""

API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

Paste your Gemini API key: ··········


In [111]:
import time

print("Testing Gemini API...")

start_time = time.time()

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Reply with only the word SUCCESS."
    )

    print("Gemini response:", response.text)
    print("Time taken:", round(time.time() - start_time, 2), "seconds")

except Exception as e:
    print("Error:", repr(e))

Testing Gemini API...
Gemini response: SUCCESS
Time taken: 29.39 seconds


In [110]:
# Step 3 — Create the CRAFT prompt and function


def analyse_note(note_text):
    prompt = f"""
Context: These are call-centre notes about missed clinic appointments.
Role: You are a careful appointment no-show analyst.
Action: Identify the most likely reason the patient missed the appointment.
Use only these root_cause values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.
If the note is unclear or does not support a listed reason, choose other.
Do not guess. Keep the summary to at most 25 words.
Tone: Clear, neutral, and concise.
Format: Return exactly the four required JSON fields:
root_cause, confidence, summary, recommended_action.

Note:
{note_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)
    return data

In [83]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
  if 'followup_notes.csv' in files:
    print(os.path.join(root,'followup_notes.csv'))

/content/drive/MyDrive/careloop_project/data/followup_notes.csv


In [85]:
# Step 4 — Find note N001 and test the function

import pandas as pd

notes = pd.read_csv("/content/drive/MyDrive/careloop_project/data/followup_notes.csv")
print(notes.columns.tolist())
print(notes.head())

['note_id', 'appointment_id', 'note_text']
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [86]:
#Step 5:  Test gemini function
note = notes.loc[notes["note_id"] == "N001"].iloc[0]
print(note["note_text"])
result = analyse_note(note["note_text"])
print(result)


Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed the appointment because they did not receive a reminder and subsequently forgot the scheduled visit.', 'recommended_action': "Reschedule the appointment and verify the patient's contact information for future reminder notifications."}


In [87]:
import time

def analyse_note_with_retry(note_text, retries=3):
    for attempt in range(retries):
        try:
            return analyse_note(note_text)

        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"Temporary error. Retrying {attempt + 1}/3...")
                time.sleep(10)
            else:
                raise

    return {"error": "Gemini unavailable after 3 attempts"}

In [88]:
# Step 6 - Test Gemini on the next 4 notes

for _, row in notes.head(5).iloc[1:].iterrows():

    print("Note ID:", row["note_id"])
    print("Note:", row["note_text"])

    result = analyse_note_with_retry(row["note_text"])

    print("Result:", result)
    print("-" * 50)

    time.sleep(2)

Note ID: N002
Note: Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
Result: {'root_cause': 'distance_or_transport', 'confidence': 1.0, 'summary': "Patient missed the appointment due to the clinic's distance and an inability to secure transportation on the morning of the visit.", 'recommended_action': "Offer telehealth options or refer to a clinic closer to the patient's residence."}
--------------------------------------------------
Note ID: N003
Note: Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
Result: {'root_cause': 'cost_or_fee', 'confidence': 1.0, 'summary': 'The patient missed the appointment due to the consultation fee being too high and inquired about potential discounts for follow-up visits.', 'recommended_action': 'Provide information on available financial assistance, payment plans, or discount policies for follow-up con

In [89]:
print("Model:", MODEL_NAME)
print("API key loaded:", bool(API_KEY))

Model: gemini-3.1-flash-lite
API key loaded: True


In [90]:
test_note = notes.loc[
    notes["note_id"] == "N003" , "note_text"
].iloc[0]

print(analyse_note_with_retry(test_note))

{'root_cause': 'cost_or_fee', 'confidence': 1.0, 'summary': 'The patient missed the appointment due to the consultation fee being too high and inquired about potential discounts for follow-up visits.', 'recommended_action': 'Provide information on available financial assistance, payment plans, or discount policies for follow-up consultations.'}


## TASK 4.2 - VALIDATE AND PROCESS

In [91]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError
import pandas as pd
import json
import time

# Allowed root causes
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Step 1: Define the Pydantic model
class FollowupInsight(BaseModel):
     root_cause: Literal[

        "forgot_or_no_reminder",

        "distance_or_transport",

        "cost_or_fee",

        "illness_or_emergency",

        "work_or_schedule",

        "other"

    ]
     confidence: float = Field(ge=0, le=1)
     summary: str = Field(min_length=5)
     recommended_action: str = Field(min_length = 5)

# Step 2: Process the first 10 notes
results = []

for _, row in notes.head(10).iterrows():

    note_id = row["note_id"]
    note_text = row["note_text"]

    # Default values
    raw_response = ""
    root_cause = ""
    confidence = None
    summary = ""
    recommended_action = ""

    # Step 3: Handle blank notes
    if pd.isna(note_text) or str(note_text).strip() == "":

        root_cause = "other"
        confidence = 0.0
        summary = "No note recorded"
        recommended_action = "Re-call patient"
        validation_status = "skipped_blank"

    else:

        try:
            # Call Gemini
            response = analyse_note_with_retry(str(note_text))

            # Preserve the raw response
            raw_response = json.dumps(response)

            # Step 4: Validate the response
            if "error" in response:
                raise ValueError(response["error"])

            validated = FollowupInsight.model_validate(response)

            # Check allowed root cause
            if validated.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError(
                    f"Invalid root_cause: {validated.root_cause}"
                )

            # Store validated results
            root_cause = validated.root_cause
            confidence = validated.confidence
            summary = validated.summary
            recommended_action = validated.recommended_action

            validation_status = "valid"

        except (ValidationError, ValueError, Exception) as e:

            # Keep invalid responses as they came back
            validation_status = "invalid"

            # Do not repair or replace the response
            if not raw_response:
                raw_response = str(e)

        # Pause between API calls
        time.sleep(13)

    # Step 5: Append every row
    results.append({
        "appointment_id": row["appointment_id"],
        "note_id": note_id,
        "root_cause": root_cause,
        "confidence": confidence,
        "summary": summary,
        "recommended_action": recommended_action,
        "validation_status": validation_status,
        "raw_response": raw_response
    })


# Step 6: Create the final DataFrame
notes_analysed = pd.DataFrame(results)

# Step 7: Save the CSV
notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("Total rows:", len(notes_analysed))
print("Total columns:", len(notes_analysed.columns))
print(notes_analysed)

Temporary error. Retrying 1/3...
Temporary error. Retrying 2/3...
Temporary error. Retrying 1/3...
Temporary error. Retrying 1/3...
Temporary error. Retrying 2/3...
Total rows: 10
Total columns: 8
  appointment_id note_id             root_cause  confidence  \
0          A0872    N001  forgot_or_no_reminder        1.00   
1          A0897    N002  distance_or_transport        1.00   
2          A0526    N003            cost_or_fee        1.00   
3          A0926    N004                  other        0.00   
4          A0534    N005   illness_or_emergency        1.00   
5          A0507    N006       work_or_schedule        1.00   
6          A0333    N007  forgot_or_no_reminder        0.95   
7          A0222    N008  distance_or_transport        1.00   
8          A0977    N009                  other        0.90   
9          A0596    N010       work_or_schedule        1.00   

                                             summary  \
0  The patient missed the appointment because the... 

## TASK 4.3 - BREAK IT ON PURPOSE

In [92]:
#STEP 1
from pydantic import ValidationError

# Test 1: Invalid confidence value
bad_response_1 = {
    "root_cause": "cost_or_fee",
    "confidence": 1.7,
    "summary": "The patient could not afford the consultation.",
    "recommended_action": "Offer a payment plan."
}

# Test 2: Missing required field
bad_response_2 = {
    "root_cause": "illness_or_emergency",
    "confidence": 0.9,
    "summary": "The patient was unwell."
}

# Validate both responses
for i, bad_response in enumerate(
    [bad_response_1, bad_response_2], start=1
):
    try:
        FollowupInsight.model_validate(bad_response)
        print(f"Test {i}: Validation passed")

    except ValidationError as e:
        print(f"Test {i}: Validation failed")
        print(e)
        print("-" * 50)

Test 1: Validation failed
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
--------------------------------------------------
Test 2: Validation failed
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'illness_o...he patient was unwell.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
--------------------------------------------------


In [95]:
# STEP 2 :

import json
import pandas as pd
from google.genai import types

# Prompt V1: Original prompt
V1 = """
Classify the patient's missed appointment reason.

Return a JSON object with exactly these fields:
root_cause, confidence, summary, recommended_action.

Use only these root_cause values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.

Patient note:
"""

# Prompt V2: Improved prompt
V2 = """
You are a clinic appointment analyst.

Read the patient note and identify the main reason
for the missed appointment.

Choose exactly one root_cause from:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.

Return only valid JSON with exactly four fields:
root_cause (string), confidence (number between 0 and 1),
summary (string), recommended_action (string).

If the note is ambiguous, choose the most likely reason
and explain the uncertainty in the summary.

Patient note:
"""


# Function to run one prompt
def compare_prompt(note_text, prompt, retries=3):
  for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt + str(note_text),
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    max_output_tokens=1000
                )
            )
            return json.loads(response.text)

        except Exception as e:
            if attempt == retries - 1:
                raise

            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))


test_notes = notes[
    notes["note_id"].isin(["N007", "N009"])
]

comparison_results = []

for _, row in test_notes.iterrows():

    note_id = row["note_id"]
    note_text = row["note_text"]

    result_v1 = compare_prompt(note_text, V1)
    result_v2 = compare_prompt(note_text, V2)

    comparison_results.append({
        "note_id": note_id,
        "V1_root_cause": result_v1["root_cause"],
        "V1_confidence": result_v1["confidence"],
        "V2_root_cause": result_v2["root_cause"],
        "V2_confidence": result_v2["confidence"],
        "V1_summary": result_v1["summary"],
        "V2_summary": result_v2["summary"]
    })

    time.sleep(5)

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)




,note_id,V1_root_cause,V1_confidence,V2_root_cause,V2_confidence,V1_summary,V2_summary
0,N007,forgot_or_no_reminder,1.00,forgot_or_no_reminder,0.95,The patient forgot the appointment and stated ...,The patient explicitly stated they forgot the ...
1,N009,other,0.85,other,0.60,The patient could not be reached after three a...,The patient was unreachable due to their phone...


In [96]:
# Save csv result in Google Drive

from google.colab import files

files.download("notes_analysed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## TASK 4.4 - RECOMMENDATION AND GITHUB


In [97]:
# Summarize the main reasons for missed appointments
root_cause_summary = (
    notes_analysed
    .groupby("root_cause")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Root Cause Summary:")
display(root_cause_summary)

# Show the recommended actions for each root cause
recommendations = (
    notes_analysed[
        ["root_cause", "recommended_action"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Recommended Actions:")
display(recommendations)

Root Cause Summary:


,root_cause,count
1,distance_or_transport,2
2,forgot_or_no_reminder,2
5,work_or_schedule,2
4,other,2
0,cost_or_fee,1
3,illness_or_emergency,1


Recommended Actions:


,root_cause,recommended_action
0,forgot_or_no_reminder,Reschedule the appointment and verify the pati...
1,distance_or_transport,Offer telehealth options or refer to a clinic ...
2,cost_or_fee,Provide information on available financial ass...
3,other,Re-call patient
4,illness_or_emergency,Reschedule appointment once the patient confir...
5,work_or_schedule,Reschedule the patient for an evening clinic s...
6,forgot_or_no_reminder,Reschedule the appointment and verify the pati...
7,distance_or_transport,Offer telehealth options or assist with arrang...
8,other,Send a follow-up letter by mail to confirm con...
9,work_or_schedule,Reschedule for a Saturday slot if available or...


In [98]:
import pandas as pd

# Load the two CSV files
notes_df = pd.read_csv("notes_analysed.csv")
appointments_df = pd.read_csv('/content/drive/MyDrive/careloop_project/data/appointments_clean.csv')

# Check the column names
print("Notes columns:")
print(notes_df.columns.tolist())

print("\nAppointments columns:")
print(appointments_df.columns.tolist())

Notes columns:
['appointment_id', 'note_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']

Appointments columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [99]:
# Merge the two datasets
merged_df = pd.merge(
    notes_df,
    appointments_df,
    on="appointment_id",
    how="left"
)

# Check the merged dataset
print("Merged dataset shape:", merged_df.shape)

display(merged_df.head())

# Cross-tabulation of root cause against reminder status
cross_tab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("Root Cause vs Reminder Sent:")
display(cross_tab)

Merged dataset shape: (10, 23)


,appointment_id,note_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0872,N001,forgot_or_no_reminder,1.0,The patient missed the appointment because the...,Reschedule the appointment and verify the pati...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi...",P144,C05,...,F,17.6,750.00,App,Consultation,No,1,No-Show,14,1
1,A0897,N002,distance_or_transport,1.0,Patient missed the appointment due to the clin...,Offer telehealth options or refer to a clinic ...,valid,"{""root_cause"": ""distance_or_transport"", ""confi...",P268,C07,...,M,17.2,1218.75,Phone,Follow-up,No,0,No-Show,5,1
2,A0526,N003,cost_or_fee,1.0,The patient missed the appointment due to the ...,Provide information on available financial ass...,valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 1....",P006,C04,...,F,5.3,800.00,App,Consultation,Yes,0,No-Show,34,1
3,A0926,N004,other,0.0,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300.00,Phone,Follow-up,No,0,No-Show,40,1
4,A0534,N005,illness_or_emergency,1.0,Patient missed the appointment due to high fev...,Reschedule appointment once the patient confir...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid...",P400,C04,...,F,4.2,550.00,App,Follow-up,Yes,0,No-Show,26,1


Root Cause vs Reminder Sent:


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


In [100]:
merged_df.to_csv("merged_notes_appointments.csv", index=False)

print("Merged dataset saved successfully!")

Merged dataset saved successfully!


In [101]:
#  Check merged dataset

print("Merged dataset shape :", merged_df.shape)

print("\nRoot cause vs reminder:")

display(pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
))

Merged dataset shape : (10, 23)

Root cause vs reminder:


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


In [102]:
%whos

Variable                  Type                      Data/Info
-------------------------------------------------------------
ALLOWED_ROOT_CAUSES       list                      n=6
API_KEY                   str                       AQ.Ab8RN6KWnySKVrCcp0LniY<...>JFjfOdCpPZOrEmdKlueda2vTQ
BaseModel                 ModelMetaclass            <class 'pydantic.main.BaseModel'>
Field                     function                  <function Field at 0x78d1b49c1d00>
FollowupInsight           ModelMetaclass            <class '__main__.FollowupInsight'>
Literal                   _TypedCacheSpecialForm    typing.Literal
MODEL_NAME                str                       gemini-3.1-flash-lite
Path                      type                      <class 'pathlib._local.Path'>
SCHEMA                    dict                      n=3
SYSTEM_TEXT               str                       \nYou classify call-centr<...> requested JSON object.\n
V1                        str                       \nClassify th

In [103]:
# Save the merged dataset
merged_df.to_csv(
    "merged_notes_appointments.csv",
    index=False
)

print("Merged dataset saved successfully!")

Merged dataset saved successfully!


In [104]:
merged_df.to_csv(
    "/content/drive/MyDrive/GenAI_Project_Results/merged_notes_appointments.csv",
    index=False
)

print("Merged dataset saved to Google Drive!")

Merged dataset saved to Google Drive!


In [105]:
display(cross_tab)

reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


Final Business Recommendation

CareLoop should first introduce a targeted reminder pilot for appointments identified as high risk by the no-show prediction model.

The pilot should use the risk score generated on the evening before the appointment, after the day’s reminders have been sent or skipped. The front desk can then contact the highest-risk patients.

The AI-extracted call-centre notes should help staff understand why patients may miss appointments and guide the type of support offered. For example, patients reporting transport difficulties may need assistance with travel planning, while patients who forgot their appointment may benefit from an additional reminder.

The pilot should be evaluated by comparing no-show rates and the number of additional reminders required. The model should support staff decisions rather than be treated as a definitive prediction.

These recommendations are preliminary because only 10 call-centre notes were analysed, and the model’s test precision was 38%.

In [108]:
from pathlib import Path

readme_content = """
# Patient Appointment No-Show Prediction

## Project Summary

Part 1: Analysed appointment data to understand patterns associated with patient no-shows.

Part 2: Prepared the dataset and created features for predicting missed appointments.

Part 3: Built a Balanced Logistic Regression model with 68.18% test accuracy and 0.742 ROC-AUC.

Part 4: Used Gemini to analyse appointment notes and identify reasons for missed appointments.

## Business Recommendation

The clinic should use the no-show prediction model to identify patients who may need additional reminders. The model identified 30 of 44 actual no-shows in the test dataset, giving a recall of 68%. However, its precision was 38%, meaning many patients flagged as high-risk would still attend. The clinic should consider the cost of unnecessary reminders and use the model as a support tool rather than relying on it as a definitive prediction.

## Limitations

1. Only 10 notes were analysed using the GenAI model, so they cannot represent all clinic-wide reasons for missed appointments.
2. The model was trained on data from before the train-test split, so its evaluation may not represent future performance.
3. The same patient may appear in both the training and test datasets.
4. The train-test split was random rather than based on appointment dates.
5. The data is observational, so the associations found do not establish causation.
"""

Path("README.md").write_text(readme_content)

print("README.md created successfully!")

README.md created successfully!


In [109]:
import os

file_path = "/content/drive/MyDrive/GenAI_Project_Results/merged_notes_appointments.csv"

print("File exists:", os.path.exists(file_path))

File exists: True


In [106]:
from google.colab import files
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>